In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import time
import math
from tqdm import tqdm
from torchmetrics.functional import calibration_error

In [2]:
# =========================================================
# DEVICE
# =========================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [3]:
# =========================================================
# DATASET
# =========================================================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

trainset = torchvision.datasets.CIFAR100(root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

testset = torchvision.datasets.CIFAR100(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2, pin_memory=True)

In [4]:
# =========================================================
# VGG16 Model for CIFAR-100
# =========================================================
class VGG16_CIFAR100(nn.Module):
    def __init__(self, num_classes=100):
        super(VGG16_CIFAR100, self).__init__()
        self.features = torchvision.models.vgg16_bn(weights=None).features
        self.classifier = nn.Sequential(
            nn.Linear(512, 512),
            nn.ReLU(True),
            #nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1,1))
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = VGG16_CIFAR100().to(device)
criterion = nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

In [5]:
# =========================================================
# METRICS
# =========================================================
def topk_accuracy(output, target, topk=(1,)):
    maxk = max(topk)
    _, pred = output.topk(maxk, 1, True, True)
    correct = pred.eq(target.view(-1,1).expand_as(pred))
    res = []
    for k in topk:
        correct_k = correct[:, :k].sum().item()
        res.append(correct_k)
    return res

def mean_off_diagonal_covariance(features, eps=1e-8):
    X = features - features.mean(dim=0, keepdim=True)
    cov = (X.T @ X) / (X.shape[0]-1)
    var = cov.diag()
    denom = torch.sqrt(var.unsqueeze(0)*var.unsqueeze(1)).clamp_min(eps)
    corr = cov/denom
    corr.fill_diagonal_(0)
    return corr.abs().mean()

def linear_CKA(X, Y):
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    hsic = torch.norm(X.T @ Y, p='fro')**2
    var1 = torch.norm(X.T @ X, p='fro') * torch.norm(Y.T @ Y, p='fro')
    return (hsic / var1).item()


In [6]:
# =========================================================
# DCD Regularizer Class
# =========================================================
class DCDRegularizer:
    def __init__(self, model, device,
                 lambda_a=1e-3, lambda_w=1e-4, lambda_s=1e-6, lambda_q=1e-5,
                 group_size=32, k=8, r=16,
                 target_layers=None,
                 use_ema=False, ema_beta=0.9,
                 q_alpha=0.1, q_apply_every=1):
        self.model = model
        self.device = device
        self.lambda_a = lambda_a
        self.lambda_w = lambda_w
        self.lambda_s = lambda_s
        self.lambda_q = lambda_q
        self.group_size = group_size
        self.k = k
        self.r = r
        self.q_alpha = q_alpha
        self.q_apply_every = q_apply_every
        self.step_counter = 0

        if target_layers is None:
            target_layers = ["features.10", "features.17", "features.24", "features.31"]
        self.target_layers = set(target_layers)

        self.activations = {}
        self.group_sketches = {}
        self.weight_proj = {}
        self.cov_ema = {}
        self.use_ema = use_ema
        self.ema_beta = ema_beta

        for name, module in self.model.named_modules():
            if name in self.target_layers and isinstance(module, nn.Conv2d):
                module.register_forward_hook(self._make_activation_hook(name))

    def _make_activation_hook(self, name):
        def hook(module, input, output):
            self.activations[name] = output.detach()
        return hook

    def _ensure_sketch_for_group(self, layer_name, g, c_g):
        key = (layer_name, g)
        if key not in self.group_sketches:
            kk = min(self.k, c_g)
            Sg = torch.randn(c_g, kk, device=self.device) / math.sqrt(kk)
            self.group_sketches[key] = Sg
            if self.use_ema:
                self.cov_ema[key] = torch.zeros(kk, kk, device=self.device)
        return self.group_sketches[key]

    def _ensure_weight_proj_for_group(self, layer_name, g, R, c_g):
        key = (layer_name, g)
        if key not in self.weight_proj:
            P = torch.randn(self.r, R, device=self.device) / math.sqrt(self.r)
            self.weight_proj[key] = P
        return self.weight_proj[key]

    def compute_L_a(self):
        L_a = torch.tensor(0.0, device=self.device)
        for name, A in self.activations.items():
            B, C, H, W = A.shape
            N = B*H*W
            X = A.permute(0,2,3,1).reshape(N,C)
            G = math.ceil(C/self.group_size)
            for g in range(G):
                start, end = g*self.group_size, min((g+1)*self.group_size, C)
                c_g = end-start
                if c_g <= 1: continue
                Xg = X[:, start:end]
                Sg = self._ensure_sketch_for_group(name,g,c_g)
                Zg = Xg @ Sg
                Zg = Zg - Zg.mean(0, keepdim=True)
                Covg = (Zg.T @ Zg) / (N-1)
                if self.use_ema:
                    key = (name,g)
                    self.cov_ema[key] = self.ema_beta*self.cov_ema[key] + (1-self.ema_beta)*Covg
                    Cov_use = self.cov_ema[key]
                else:
                    Cov_use = Covg
                offdiag = Cov_use - torch.diag(torch.diag(Cov_use))
                L_a += torch.sum(offdiag**2)
        return L_a

    def compute_L_w_and_Ls_and_Lq(self):
        L_w = torch.tensor(0.0, device=self.device)
        L_s = torch.tensor(0.0, device=self.device)
        L_q = torch.tensor(0.0, device=self.device)
        for name,module in self.model.named_modules():
            if isinstance(module, nn.Conv2d):
                W = module.weight
                C_out, C_in, kh, kw = W.shape
                R = C_in*kh*kw
                W_flat = W.view(C_out,R)
                L_s += torch.sum(torch.abs(W_flat))
                if C_out <= 1: continue
                G = math.ceil(C_out/self.group_size)
                for g in range(G):
                    start,end = g*self.group_size, min((g+1)*self.group_size,C_out)
                    c_g = end-start
                    if c_g <= 1: continue
                    Wg = W_flat[start:end,:]
                    P = self._ensure_weight_proj_for_group(name,g,R,c_g)
                    Wg_proj = (P @ Wg.T).T
                    M = Wg_proj @ Wg_proj.T
                    Icg = torch.eye(c_g, device=self.device)
                    L_w += torch.sum((M-Icg)**2)
                    if self.lambda_q>0 and (self.step_counter % max(1,self.q_apply_every)==0):
                        s = Wg_proj.abs().mean().clamp_min(1e-6)
                        x = Wg_proj / s
                        rounded = torch.round(x)
                        soft_round_dist = (x - torch.tanh(self.q_alpha*(x-rounded)))**2
                        L_q += torch.sum(soft_round_dist)
        return L_w,L_s,L_q

    def __call__(self):
        self.step_counter += 1
        L_a = self.compute_L_a()
        L_w,L_s,L_q = self.compute_L_w_and_Ls_and_Lq()
        return L_a,L_w,L_s,L_q

In [7]:
# =========================================================
# DCD Instance
# =========================================================
dcd = DCDRegularizer(model, device,
                     lambda_a=1e-5, lambda_w=1e-6, lambda_s=1e-7, lambda_q=0,
                     group_size=32, k=8, r=16,
                     target_layers=["features.10","features.17","features.24","features.31"],
                     use_ema=True, ema_beta=0.95)

lambda_a = dcd.lambda_a
lambda_w = dcd.lambda_w
lambda_s = dcd.lambda_s
lambda_q = dcd.lambda_q


In [8]:
# =========================================================
# TRAINING LOOP
# =========================================================
num_epochs = 80
best_acc = 0
best_val_loss = float('inf')
convergence_epoch = None
patience = 8
no_improve_epochs = 0
train_losses, test_losses = [], []
start_time = time.time()

for epoch in range(num_epochs):
    model.train()
    running_loss, total, correct_top1, correct_top5 = 0.0, 0, 0, 0
    epoch_start = time.time()
    dcd.activations = {}

    for inputs, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        task_loss = criterion(outputs, labels)
        L_a,L_w,L_s,L_q = dcd()
        total_loss = task_loss + lambda_a*L_a + lambda_w*L_w + lambda_s*L_s + lambda_q*L_q
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        running_loss += total_loss.item()
        top1,top5 = topk_accuracy(outputs, labels, topk=(1,5))
        correct_top1 += top1
        correct_top5 += top5
        total += labels.size(0)

    train_loss = running_loss/len(trainloader)
    train_acc1 = 100*correct_top1/total
    train_acc5 = 100*correct_top5/total
    train_losses.append(train_loss)

    # Validation
    model.eval()
    correct_top1, correct_top5, total = 0,0,0
    test_loss = 0.0
    probs, targets, features = [],[],[]

    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            top1,top5 = topk_accuracy(outputs, labels, topk=(1,5))
            correct_top1 += top1
            correct_top5 += top5
            total += labels.size(0)
            probs.append(F.softmax(outputs, dim=1).cpu())
            targets.append(labels.cpu())
            feats = model.features(inputs).view(inputs.size(0), -1).cpu()
            features.append(feats)

    test_loss /= len(testloader)
    test_acc1 = 100*correct_top1/total
    test_acc5 = 100*correct_top5/total
    train_test_gap = abs(train_acc1 - test_acc1)
    all_probs = torch.cat(probs)
    all_targets = torch.cat(targets)
    ece = calibration_error(all_probs, all_targets, n_bins=15, task="multiclass", num_classes=100).item()
    all_features = torch.cat(features, dim=0)
    moc_val = mean_off_diagonal_covariance(all_features).item()

    try:
        L_a_val = float(L_a.detach().cpu().item())
        L_w_val = float(L_w.detach().cpu().item())
        L_s_val = float(L_s.detach().cpu().item())
        L_q_val = float(L_q.detach().cpu().item())
    except:
        L_a_val = L_w_val = L_s_val = L_q_val = float('nan')

    print(f"\n📊 Epoch {epoch+1}:")
    print(f"Train Loss: {train_loss:.6f} | Train Acc@1: {train_acc1:.2f}% | Train Acc@5: {train_acc5:.2f}%")
    print(f"Val Loss: {test_loss:.6f} | Val Acc@1: {test_acc1:.2f}% | Val Acc@5: {test_acc5:.2f}%")
    print(f"Train-Test Gap: {train_test_gap:.2f}% | ECE: {ece:.4f} | MOC: {moc_val:.6f}")
    print(f"DCD λ*L: {lambda_a*L_a_val:.6e}, {lambda_w*L_w_val:.6e}, {lambda_s*L_s_val:.6e}, {lambda_q*L_q_val:.6e}")
    print(f"L components: L_a={L_a_val:.6e}, L_w={L_w_val:.6e}, L_s={L_s_val:.6e}, L_q={L_q_val:.6e}")
    print(f"⏱ Epoch Time: {time.time()-epoch_start:.2f}s")

    if test_loss < best_val_loss - 1e-4:
        best_val_loss = test_loss
        no_improve_epochs = 0
    else:
        no_improve_epochs += 1
        if no_improve_epochs >= patience:
            print("\n⏹ Early stopping triggered.")
            break

    if test_acc1 > best_acc:
        best_acc = test_acc1
        convergence_epoch = epoch+1
        torch.save(model.state_dict(), "best_vgg16_cifar100_dcd.pth")

total_time = time.time() - start_time
print(f"\n✅ Training Complete. Best Top-1 Accuracy: {best_acc:.2f}% at Epoch {convergence_epoch}")
print(f"Total Training Time: {total_time/60:.2f} minutes")


Epoch 1/80: 100%|██████████| 391/391 [00:32<00:00, 12.05it/s]



📊 Epoch 1:
Train Loss: 4.075731 | Train Acc@1: 7.49% | Train Acc@5: 25.90%
Val Loss: 3.984338 | Val Acc@1: 8.40% | Val Acc@5: 29.23%
Train-Test Gap: 0.91% | ECE: 0.0717 | MOC: 0.336634
DCD λ*L: 2.602137e-02, 7.263548e-03, 2.493374e-02, 0.000000e+00
L components: L_a=2.602137e+03, L_w=7.263548e+03, L_s=2.493374e+05, L_q=0.000000e+00
⏱ Epoch Time: 34.04s


Epoch 2/80: 100%|██████████| 391/391 [00:32<00:00, 11.96it/s]



📊 Epoch 2:
Train Loss: 3.608664 | Train Acc@1: 13.98% | Train Acc@5: 39.95%
Val Loss: 3.647736 | Val Acc@1: 12.77% | Val Acc@5: 40.14%
Train-Test Gap: 1.21% | ECE: 0.0909 | MOC: 0.307184
DCD λ*L: 2.372185e-02, 4.144827e-03, 2.389772e-02, 0.000000e+00
L components: L_a=2.372185e+03, L_w=4.144827e+03, L_s=2.389772e+05, L_q=0.000000e+00
⏱ Epoch Time: 34.08s


Epoch 3/80: 100%|██████████| 391/391 [00:35<00:00, 11.08it/s]


...


📊 Epoch 20:
Train Loss: 1.402887 | Train Acc@1: 60.30% | Train Acc@5: 87.97%
Val Loss: 2.295776 | Val Acc@1: 42.59% | Val Acc@5: 73.73%
Train-Test Gap: 17.71% | ECE: 0.1263 | MOC: 0.172661
DCD λ*L: 6.273631e-04, 3.092144e-03, 1.480332e-02, 0.000000e+00
L components: L_a=6.273631e+01, L_w=3.092144e+03, L_s=1.480332e+05, L_q=0.000000e+00
⏱ Epoch Time: 34.82s


Epoch 21/80: 100%|██████████| 391/391 [00:35<00:00, 11.17it/s]



📊 Epoch 21:
Train Loss: 1.370360 | Train Acc@1: 61.04% | Train Acc@5: 88.57%
Val Loss: 2.218116 | Val Acc@1: 44.90% | Val Acc@5: 74.31%
Train-Test Gap: 16.14% | ECE: 0.1061 | MOC: 0.165773
DCD λ*L: 5.215576e-04, 3.120498e-03, 1.462584e-02, 0.000000e+00
L components: L_a=5.215576e+01, L_w=3.120498e+03, L_s=1.462584e+05, L_q=0.000000e+00
⏱ Epoch Time: 36.55s


Epoch 22/80: 100%|██████████| 391/391 [00:33<00:00, 11.77it/s]



📊 Epoch 22:
Train Loss: 1.333860 | Train Acc@1: 62.42% | Train Acc@5: 89.00%
Val Loss: 2.387585 | Val Acc@1: 41.27% | Val Acc@5: 72.84%
Train-Test Gap: 21.15% | ECE: 0.1513 | MOC: 0.169384
DCD λ*L: 4.901738e-04, 3.104560e-03, 1.446543e-02, 0.000000e+00
L components: L_a=4.901738e+01, L_w=3.104560e+03, L_s=1.446543e+05, L_q=0.000000e+00
⏱ Epoch Time: 34.60s

⏹ Early stopping triggered.

✅ Training Complete. Best Top-1 Accuracy: 53.43% at Epoch 14
Total Training Time: 12.83 minutes
